# V8 PIR Pipeline: The 'Recall & Precision' Engine
Enhanced Candidate Generation + Advanced Feature Engineering + 3-Fold Time-Aware CV
Targeting Precision@10 > 0.25


In [1]:
import polars as pl
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix
import optuna
import gc
import warnings
import re
warnings.filterwarnings('ignore')

T_PATH = '/kaggle/input/datasets/kinonquc/qkindataset2/transaction_full_2025.parquet'
I_PATH = '/kaggle/input/datasets/kinonquc/qkindataset2/items.parquet'


In [2]:
print("=== LOADING & PRE-PROCESSING DATA ===")
df_raw = pl.read_parquet(T_PATH).select([
    pl.col('customer_id').cast(pl.Int64),
    pl.col('item_id').cast(pl.Utf8),
    pl.col('updated_date').cast(pl.Datetime).alias('event_ts'),
    pl.col('location').cast(pl.Utf8),
    pl.col('price').cast(pl.Float32),
    pl.col('quantity').cast(pl.Float32)
]).with_columns(pl.col('event_ts').dt.month().alias('month'))

items_df = pl.read_parquet(I_PATH).select([
    'item_id', 'category', 'category_l1', 'category_l2', 'category_l3', 'brand', 'size'
]).with_columns(pl.col('item_id').cast(pl.Utf8))

cat_cols = ['category', 'category_l1', 'category_l2', 'category_l3', 'brand']
for c in cat_cols:
    items_df = items_df.with_columns(pl.col(c).fill_null('Unknown').cast(pl.Categorical).to_physical().cast(pl.Int32).alias(f'{c}_id'))

def standardize_age(val):
    if not isinstance(val, str): return -1.0
    val = val.lower().strip()
    match = re.search(r'(\d+)\s*(m|y|tháng|tuổi)', val)
    if match:
        num = float(match.group(1))
        unit = match.group(2)
        if unit in ['m', 'tháng']: return num / 12.0
        return num
    return -1.0

size_map = {row[0]: standardize_age(row[1]) for row in items_df.select(['item_id', 'size']).iter_rows()}
items_df = items_df.with_columns(pl.col('item_id').replace(size_map, default=-1.0).cast(pl.Float32).alias('item_age_proxy'))


=== LOADING & PRE-PROCESSING DATA ===


In [3]:
print("=== V8 RETRIEVER: ENHANCED CANDIDATE GENERATION ===")
class V8Retriever:
    def __init__(self, history_df, items_df):
        self.history_df = history_df
        self.items_df = items_df
        self.max_ts = history_df['event_ts'].max()
        
        # Metadata caching
        self.item_locs = history_df.group_by('item_id').agg(pl.col('location').unique().alias('item_hubs'))
        self.item_prcs = history_df.group_by('item_id').agg(pl.col('price').median().alias('item_p'))
        self._build_cf_indexes()
        self._build_trend_indexes()

    def _build_cf_indexes(self):
        print(" Building SVD & I2I (Increased Capacity)...")
        # Time-weighted interactions
        hist = self.history_df.group_by(['customer_id', 'item_id']).agg([
            pl.col('quantity').sum().alias('w'),
            pl.col('event_ts').max().alias('last_ts')
        ])
        days_diff = (self.max_ts - pl.col('last_ts')).dt.total_days()
        hist = hist.with_columns((pl.col('w') * (0.95 ** (days_diff / 30.0))).alias('w'))
        
        hist = hist.with_columns([
            pl.col('customer_id').rank('dense').cast(pl.Int64).alias('u_idx') - 1,
            pl.col('item_id').rank('dense').cast(pl.Int32).alias('i_idx') - 1
        ])
        self.u_map = hist.select(['customer_id', 'u_idx']).unique()
        self.i_map = hist.select(['item_id', 'i_idx']).unique()
        self.u2idx = dict(zip(self.u_map['customer_id'], self.u_map['u_idx']))
        self.items_list = self.i_map.sort('i_idx')['item_id'].to_list()
        
        self.matrix = csr_matrix((hist['w'].to_numpy(), (hist['u_idx'].to_numpy(), hist['i_idx'].to_numpy())), 
                                shape=(self.u_map.height, self.i_map.height), dtype=np.float32)
        
        # SVD - Latent interests
        self.svd = TruncatedSVD(n_components=160, random_state=42) # Increased from 128
        self.u_vecs = self.svd.fit_transform(self.matrix)
        self.i_vecs = self.svd.components_.T
        
        # I2I - Co-purchase strength
        norm_m = normalize(self.matrix, norm='l2', axis=0)
        self.i2i_sim = (norm_m.T.dot(norm_m)).astype(np.float32)
        self.i2i_sim.setdiag(0)

    def _build_trend_indexes(self):
        print(" Building Multi-Level Trend Indexes...")
        # Hub-specific trends (Last 60 days)
        self.local_trend = self.history_df.filter(pl.col('event_ts') >= self.max_ts - pl.duration(days=60))\
            .group_by(['location', 'item_id']).len()\
            .sort(['location', 'len'], descending=[False, True])\
            .group_by('location').head(100)
        
        # Global Hot items (Last 14 days - faster turnover)
        self.global_hot = self.history_df.filter(pl.col('event_ts') >= self.max_ts - pl.duration(days=14))\
            .group_by('item_id').len().sort('len', descending=True).head(100).select('item_id')

        # Category/Brand affinity sources
        j_df = self.history_df.filter(pl.col('event_ts') >= self.max_ts - pl.duration(days=90))\
            .join(self.items_df.select(['item_id', 'category_l1', 'brand']), on='item_id')
        
        self.cat_trend = j_df.group_by(['category_l1', 'item_id']).len()\
            .sort(['category_l1', 'len'], descending=[False, True])\
            .group_by('category_l1').head(40)
            
        self.brand_trend = j_df.filter(pl.col('brand') != 'Không xác định')\
            .group_by(['brand', 'item_id']).len()\
            .sort(['brand', 'len'], descending=[False, True])\
            .group_by('brand').head(15)

    def get_candidates(self, target_users):
        hist_s = self.history_df.filter(pl.col('customer_id').is_in(target_users))
        cands = {}
        
        # 1. History / Repurchase
        cands['rep'] = hist_s.select(['customer_id', 'item_id']).unique()
        
        # 2. Global Hot
        cands['hot'] = pl.DataFrame({'customer_id': target_users}).join(self.global_hot.with_columns(pl.lit(1).alias('_k')), how='cross').drop('_k')
        
        # 3. Local Trends
        user_loc = hist_s.group_by('customer_id').agg(pl.col('location').mode().first().alias('location'))
        cands['local'] = user_loc.join(self.local_trend, on='location').select(['customer_id', 'item_id']).unique()
        
        # 4. Category/Brand Affinity
        user_cats = hist_s.join(self.items_df.select(['item_id', 'category_l1']), on='item_id')\
            .group_by(['customer_id', 'category_l1']).len()\
            .sort(['customer_id', 'len'], descending=[False, True]).group_by('customer_id').head(3)
        cands['cat'] = user_cats.join(self.cat_trend, on='category_l1').select(['customer_id', 'item_id']).unique()
        
        # 5. Collaborative Filtering
        t_idx = [self.u2idx[u] for u in target_users if u in self.u2idx]
        t_u = [u for u in target_users if u in self.u2idx]
        i_arr = np.array(self.items_list)
        chunk = 2000
        c_i2i, c_svd = [], []
        for i in range(0, len(t_idx), chunk):
            idx = t_idx[i:i+chunk]
            u_b = np.array(t_u[i:i+chunk])
            # SVD scores
            s_s = self.u_vecs[idx].dot(self.i_vecs.T)
            t50 = np.argsort(-s_s, axis=1)[:, :50]
            c_svd.append(pl.DataFrame({'customer_id': pl.Series(np.repeat(u_b, 50), dtype=pl.Int64), 'item_id': i_arr[t50.flatten()]}))
            # I2I scores
            s_i = self.matrix[idx].dot(self.i2i_sim).toarray()
            t60 = np.argsort(-s_i, axis=1)[:, :60]
            mask = np.take_along_axis(s_i, t60, axis=1) > 0
            c_i2i.append(pl.DataFrame({'customer_id': pl.Series(np.repeat(u_b, 60)[mask.flatten()], dtype=pl.Int64), 
                                      'item_id': i_arr[t60.flatten()][mask.flatten()]}))
        
        cands['svd'] = pl.concat(c_svd).unique() if c_svd else pl.DataFrame(schema={'customer_id': pl.Int64, 'item_id': pl.Utf8})
        cands['i2i'] = pl.concat(c_i2i).unique() if c_i2i else pl.DataFrame(schema={'customer_id': pl.Int64, 'item_id': pl.Utf8})
        
        # Combine and apply Hard Filters (Hub + Price)
        all_cands = pl.concat([df for df in cands.values() if df.height > 0]).unique()
        
        user_prof = hist_s.group_by('customer_id').agg([
            pl.col('location').mode().first().alias('loc'),
            pl.col('price').mean().alias('avg_p')
        ])
        
        # Hub Filter: Item must have been sold in the user's hub
        item_loc_flat = self.item_locs.explode('item_hubs').rename({'item_hubs': 'loc'})
        f = all_cands.join(user_prof, on='customer_id', how='left')\
            .join(item_loc_flat, on=['item_id', 'loc'], how='inner')\
            .join(self.item_prcs, on='item_id', how='left')\
            .filter((pl.col('item_p') <= pl.col('avg_p') * 8) | (pl.col('avg_p').is_null()))\
            .select(['customer_id', 'item_id'])
            
        return f


=== V8 RETRIEVER: ENHANCED CANDIDATE GENERATION ===


In [4]:
def create_dataset_v8(history_df, truth_df, items_df, sample_users=None, n_negatives=100):
    if sample_users:
        valid_u = history_df['customer_id'].unique().shuffle(seed=42).head(sample_users).to_list()
    else:
        valid_u = history_df['customer_id'].unique().to_list()
    
    hist_s = history_df.filter(pl.col('customer_id').is_in(valid_u))
    retriever = V8Retriever(history_df, items_df)
    filtered_cands = retriever.get_candidates(valid_u)
    
    if truth_df is not None:
        truth = truth_df.filter(pl.col('customer_id').is_in(valid_u)).select(['customer_id', 'item_id']).unique()
        ds = filtered_cands.join(truth.with_columns(pl.lit(1).cast(pl.Int8).alias('target')), on=['customer_id', 'item_id'], how='left').fill_null(0)
        missed = truth.join(filtered_cands, on=['customer_id', 'item_id'], how='anti').with_columns(pl.lit(1).cast(pl.Int8).alias('target'))
        ds = pl.concat([ds, missed]).unique(subset=['customer_id', 'item_id'])
        if n_negatives:
            pos = ds.filter(pl.col('target') == 1)
            neg = ds.filter(pl.col('target') == 0).sample(fraction=1.0, shuffle=True, seed=42).group_by('customer_id').head(n_negatives)
            ds = pl.concat([pos, neg]).sort(['customer_id', 'target'], descending=[False, True])
    else:
        ds = filtered_cands
    
    # --- Feature Engineering ---
    max_ts = history_df['event_ts'].max()
    
    # 1. User Profile Features
    u_prof = history_df.group_by('customer_id').agg([
        pl.col('item_id').n_unique().alias('u_unique_items'),
        pl.col('quantity').sum().alias('u_total_qty'),
        (max_ts - pl.col('event_ts').min()).dt.total_days().alias('u_tenure_days'),
        pl.col('price').mean().alias('u_avg_price'),
        pl.col('price').std().alias('u_std_price'),
        (pl.col('item_id').n_unique() / pl.col('quantity').sum().clip(1)).alias('u_exploration_ratio')
    ])
    
    # 2. Item Profile Features
    i_prof = history_df.group_by('item_id').agg([
        pl.col('customer_id').n_unique().alias('i_unique_users'),
        pl.col('quantity').sum().alias('i_total_qty'),
        pl.col('location').n_unique().alias('i_hubs_count')
    ])
    
    # 3. User-Item Interaction Features
    ui_hist = hist_s.group_by(['customer_id', 'item_id']).agg([
        pl.col('quantity').sum().alias('ui_total_qty'),
        pl.col('event_ts').max().alias('ui_last_buy_ts'),
        pl.col('quantity').len().alias('ui_buy_freq')
    ]).with_columns((max_ts - pl.col('ui_last_buy_ts')).dt.total_days().alias('ui_recency_days'))
    
    # 4. Momentum / Velocity
    vol_7d = history_df.filter(pl.col('event_ts') >= max_ts - pl.duration(days=7)).group_by('item_id').len().rename({'len': 'v7'})
    vol_21d = history_df.filter(pl.col('event_ts') >= max_ts - pl.duration(days=21)).group_by('item_id').len().rename({'len': 'v21'})
    momentum = vol_7d.join(vol_21d, on='item_id', how='left').with_columns((pl.col('v7') / (pl.col('v21') / 3.0 + 1)).alias('item_momentum'))
    
    # 5. User-Category Affinity
    u_cat_pref = history_df.join(items_df.select(['item_id', 'category_l1']), on='item_id')\
        .group_by(['customer_id', 'category_l1']).len()\
        .with_columns((pl.col('len') / pl.col('len').sum().over('customer_id')).alias('u_cat_affinity'))
    
    # Joining all features
    ds = ds.join(u_prof, on='customer_id', how='left')
    ds = ds.join(i_prof, on='item_id', how='left')
    ds = ds.join(ui_hist.drop('ui_last_buy_ts'), on=['customer_id', 'item_id'], how='left')
    ds = ds.join(items_df.select(['item_id', 'item_age_proxy'] + cat_cols + [f'{c}_id' for c in cat_cols]), on='item_id', how='left')
    ds = ds.join(momentum.select(['item_id', 'item_momentum']), on='item_id', how='left')
    ds = ds.join(u_cat_pref.select(['customer_id', 'category_l1', 'u_cat_affinity']), on=['customer_id', 'category_l1'], how='left')
    
    return ds.fill_null(0)


In [5]:
print("=== V8 TIME-AWARE CROSS-VALIDATION (3-FOLD) ===")
# Fold 1: Train <= M7, Validate M8
# Fold 2: Train <= M8, Validate M9
# Fold 3: Train <= M9, Validate M10 (Primary Validation)
def get_fold(train_end, val_m):
    h = df_raw.filter(pl.col('month') <= train_end)
    t = df_raw.filter(pl.col('month') == val_m)
    return create_dataset_v8(h, t, items_df, sample_users=45000, n_negatives=100)

fold1 = get_fold(7, 8)
fold2 = get_fold(8, 9)
fold3 = get_fold(9, 10)
test_set = create_dataset_v8(df_raw.filter(pl.col('month') <= 10), df_raw.filter(pl.col('month') == 11), items_df, sample_users=25000, n_negatives=None)


=== V8 TIME-AWARE CROSS-VALIDATION (3-FOLD) ===
 Building SVD & I2I (Increased Capacity)...
 Building Multi-Level Trend Indexes...
 Building SVD & I2I (Increased Capacity)...
 Building Multi-Level Trend Indexes...
 Building SVD & I2I (Increased Capacity)...
 Building Multi-Level Trend Indexes...
 Building SVD & I2I (Increased Capacity)...
 Building Multi-Level Trend Indexes...


In [6]:
print("=== OPTUNA CV TUNING & TRAINING ===")
cat_feat_ids = [f'{c}_id' for c in cat_cols]
all_feats = [
    'u_unique_items', 'u_total_qty', 'u_tenure_days', 'u_avg_price', 'u_std_price', 'u_exploration_ratio',
    'i_unique_users', 'i_total_qty', 'i_hubs_count',
    'ui_total_qty', 'ui_recency_days', 'ui_buy_freq',
    'item_momentum', 'item_age_proxy', 'u_cat_affinity'
] + cat_feat_ids

def prep_lgb(df):
    p = df.to_pandas()
    for c in cat_feat_ids: p[c] = p[c].astype('category')
    return p[all_feats], p['target'], p.groupby('customer_id').size().values

X1, y1, g1 = prep_lgb(fold1)
X2, y2, g2 = prep_lgb(fold2)
X3, y3, g3 = prep_lgb(fold3)

def objective(trial):
    param = {
        'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [10], 'verbosity': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08),
        'num_leaves': trial.suggest_int('num_leaves', 63, 511),
        'max_depth': trial.suggest_int('max_depth', 7, 16),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 50, 200),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
    }
    # Using Fold 3 as the primary validation, trained on Folds 1 & 2
    X_train = np.concatenate([X1, X2])
    y_train = np.concatenate([y1, y2])
    g_train = np.concatenate([g1, g2])
    
    dtrain = lgb.Dataset(X_train, y_train, group=g_train)
    dval = lgb.Dataset(X3, y3, group=g3, reference=dtrain)
    m = lgb.train(param, dtrain, valid_sets=[dval], num_boost_round=400, callbacks=[lgb.early_stopping(40)])
    return m.best_score['valid_0']['ndcg@10']

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
best_params = study.best_params
print("Best Params (CV):", best_params)

# Final Training on all 3 Folds
X_final = np.concatenate([X1, X2, X3])
y_final = np.concatenate([y1, y2, y3])
g_final = np.concatenate([g1, g2, g3])
d_final = lgb.Dataset(X_final, y_final, group=g_final)
best_params.update({'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [10]})
lgb_m = lgb.train(best_params, d_final, num_boost_round=1000)


=== OPTUNA CV TUNING & TRAINING ===


[I 2026-05-16 06:58:44,000] A new study created in memory with name: no-name-910fa050-e431-4a21-b513-93527e3358db


Training until validation scores don't improve for 40 rounds
Did not meet early stopping. Best iteration is:
[397]	valid_0's ndcg@10: 0.8942


[I 2026-05-16 07:08:12,816] Trial 0 finished with value: 0.8942000745487652 and parameters: {'learning_rate': 0.022957098201940154, 'num_leaves': 183, 'max_depth': 14, 'min_data_in_leaf': 163, 'colsample_bytree': 0.5426812949470854, 'lambda_l1': 4.438508014158548e-05, 'lambda_l2': 4.320902539293132}. Best is trial 0 with value: 0.8942000745487652.


Training until validation scores don't improve for 40 rounds


...

Training until validation scores don't improve for 40 rounds
Early stopping, best iteration is:
[209]	valid_0's ndcg@10: 0.89528


[I 2026-05-16 10:26:32,057] Trial 28 finished with value: 0.895279505763479 and parameters: {'learning_rate': 0.031241680281263014, 'num_leaves': 252, 'max_depth': 12, 'min_data_in_leaf': 159, 'colsample_bytree': 0.6719201975824339, 'lambda_l1': 5.082578674753778e-07, 'lambda_l2': 7.330425216168509e-07}. Best is trial 14 with value: 0.8953119397659792.


Training until validation scores don't improve for 40 rounds
Early stopping, best iteration is:
[268]	valid_0's ndcg@10: 0.895099


[I 2026-05-16 10:34:35,533] Trial 29 finished with value: 0.8950987222757666 and parameters: {'learning_rate': 0.024271826517482848, 'num_leaves': 256, 'max_depth': 14, 'min_data_in_leaf': 171, 'colsample_bytree': 0.6523760141547277, 'lambda_l1': 6.370494889388786e-08, 'lambda_l2': 5.527614359665176e-06}. Best is trial 14 with value: 0.8953119397659792.


Best Params (CV): {'learning_rate': 0.015399694299542518, 'num_leaves': 422, 'max_depth': 16, 'min_data_in_leaf': 137, 'colsample_bytree': 0.7342318305626618, 'lambda_l1': 1.705507016402405e-06, 'lambda_l2': 0.018589308369205065}


In [7]:
print("=== FINAL EVALUATION (MONTH 11) ===")
X_ts, y_ts, _ = prep_lgb(test_set)
test_set = test_set.with_columns(pl.Series(name='pred', values=lgb_m.predict(X_ts)))

def evaluate(model_col):
    top10 = test_set.sort(['customer_id', model_col], descending=[False, True]).group_by('customer_id', maintain_order=True).head(10)
    truth_map = df_raw.filter(pl.col('month') == 11).filter(pl.col('customer_id').is_in(top10['customer_id'].unique().to_list())).group_by('customer_id').agg(pl.col('item_id'))
    truth_dict = {row[0]: set(row[1]) for row in truth_map.iter_rows()}
    pred_dict = {row[0]: list(row[1]) for row in top10.group_by('customer_id').agg(pl.col('item_id')).iter_rows()}
    h, m, p = 0, 0.0, 0.0
    for uid, truth in truth_dict.items():
        preds = pred_dict.get(uid, [])
        hits = [pr for pr in preds if pr in truth]
        h += len(hits); p += len(hits)/10.0
        for i, pr in enumerate(preds):
            if pr in truth: m += 1.0/(i+1); break
    n = max(1, len(truth_dict))
    return {'Hits': h, 'Precision@10': p/n, 'MRR': m/n}

print(evaluate('pred'))


=== FINAL EVALUATION (MONTH 11) ===
{'Hits': 12146, 'Precision@10': 0.20506500084416907, 'MRR': 0.6432506183856873}
